# MNIST Rotation Dataset — Creation & Analysis

This notebook creates the augmented MNIST rotation dataset described in the thesis (Section 4.2.1)  
and provides analysis of how many images were added per digit and per rotation angle.

**Three-stage pipeline:**
1. Train a CNN classifier on original MNIST (>99% accuracy)
2. Test candidate rotation angles; accept rotated images where classifier confidence > 0.9999
3. Apply a final uniform random rotation θ ~ Uniform(0°, 360°) to every entry

**Two variants:**
- **Thesis-exact**: candidate angles = {90°, 180°, 270°}
- **Every-10-degrees**: candidate angles = {10°, 20°, ..., 350°}

## 0. Imports & Setup

In [ ]:
import math, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR   = './data'
OUTPUT_DIR = './data/mnist_rotation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.9999
CLASSIFIER_MEAN = 0.1307
CLASSIFIER_STD  = 0.3081

print(f'Device: {device}')
print(f'Output dir: {OUTPUT_DIR}')

---
## 1. Load Raw MNIST

In [ ]:
raw_ds = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True,
                                     transform=transforms.ToTensor())
raw_loader = DataLoader(raw_ds, batch_size=4096, shuffle=False, num_workers=2)

all_imgs, all_lbls = [], []
for imgs, lbls in tqdm(raw_loader, desc='Loading MNIST'):
    all_imgs.append(imgs)
    all_lbls.append(lbls)

raw_images = torch.cat(all_imgs, dim=0)  # [60000, 1, 28, 28]  in [0,1]
raw_labels = torch.cat(all_lbls, dim=0)  # [60000]

print(f'Loaded {raw_images.shape[0]:,} images  shape={raw_images.shape}')
print(f'Label distribution:')
for d in range(10):
    print(f'  Digit {d}: {(raw_labels==d).sum().item():,}')

---
## 2. Stage 1 — Train CNN Classifier

Architecture from thesis Appendix 6.2.1: three conv blocks + FC layers.  
Hyperparameters: AdamW lr=1e-3, weight_decay=1e-4, ReduceLROnPlateau(factor=0.5, patience=2), 15 epochs.

In [ ]:
class MNISTClassifier(nn.Module):
    """CNN classifier — thesis Appendix 6.2.1 / Figure 6.9."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),               # 28→14
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),               # 14→7
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*7*7, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.block3(self.block2(self.block1(x))))

In [ ]:
CLASSIFIER_PATH = os.path.join(OUTPUT_DIR, 'mnist_classifier.pt')
NEPOCHS_CLASSIFIER = 15

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize([CLASSIFIER_MEAN], [CLASSIFIER_STD]),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([CLASSIFIER_MEAN], [CLASSIFIER_STD]),
])

train_cls_ds = torchvision.datasets.MNIST(DATA_DIR, train=True,  transform=train_transform)
test_cls_ds  = torchvision.datasets.MNIST(DATA_DIR, train=False, transform=test_transform)
train_cls_loader = DataLoader(train_cls_ds, batch_size=128, shuffle=True,  num_workers=2)
test_cls_loader  = DataLoader(test_cls_ds,  batch_size=256, shuffle=False, num_workers=2)

if os.path.exists(CLASSIFIER_PATH):
    print(f'Loading existing classifier from {CLASSIFIER_PATH}')
    classifier = MNISTClassifier().to(device)
    classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location=device))
    classifier.eval()
else:
    classifier = MNISTClassifier().to(device)
    optimizer  = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    criterion  = nn.CrossEntropyLoss()

    train_losses, val_accs = [], []
    for epoch in range(1, NEPOCHS_CLASSIFIER+1):
        classifier.train()
        ep_loss = 0
        for imgs, lbls in tqdm(train_cls_loader, desc=f'Epoch {epoch}/{NEPOCHS_CLASSIFIER}', leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(classifier(imgs), lbls)
            loss.backward(); optimizer.step()
            ep_loss += loss.item()

        classifier.eval()
        correct = total = val_loss = 0
        with torch.no_grad():
            for imgs, lbls in test_cls_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = classifier(imgs)
                val_loss += criterion(logits, lbls).item()
                correct  += (logits.argmax(1) == lbls).sum().item()
                total    += lbls.size(0)
        acc = correct / total
        scheduler.step(val_loss)
        val_accs.append(acc)
        train_losses.append(ep_loss / len(train_cls_loader))
        print(f'  Epoch {epoch:2d} | val_acc={acc:.4f} | lr={optimizer.param_groups[0]["lr"]:.2e}')

    torch.save(classifier.state_dict(), CLASSIFIER_PATH)
    print(f'\nClassifier saved to {CLASSIFIER_PATH}')
    classifier.eval()

In [ ]:
# Quick accuracy check
classifier.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in test_cls_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        correct += (classifier(imgs).argmax(1) == lbls).sum().item()
        total   += lbls.size(0)
print(f'Test accuracy: {correct/total*100:.2f}%  ({correct}/{total})')

---
## 3. Stage 2 — Logical Angle Assignment

For each image, rotate by candidate angles and accept those where classifier confidence > 0.9999.

In [ ]:
@torch.no_grad()
def run_stage2(raw_images, raw_labels, classifier, candidate_angles_deg,
               confidence_threshold=0.9999, batch_size=512):
    """
    Returns:
      aug_images:      [M, 1, 28, 28]  float [0,1]
      aug_labels:      [M]             predicted digit
      aug_base_angles: [M]             discrete rotation angle (0 = original)
      stats:           dict  angle -> per-digit counts   (for analysis)
    """
    classifier.eval()
    N = raw_images.shape[0]

    all_imgs, all_lbls, all_base = [], [], []
    # Always include originals
    all_imgs.append(raw_images)
    all_lbls.append(raw_labels)
    all_base.append(torch.zeros(N, dtype=torch.float32))

    # per_angle_per_digit[angle][digit] = count
    stats = {a: [0]*10 for a in candidate_angles_deg}

    for angle_deg in tqdm(candidate_angles_deg, desc='Stage 2: angle scan'):
        a_imgs, a_lbls, a_base = [], [], []
        for s in range(0, N, batch_size):
            e    = min(s + batch_size, N)
            imgs = raw_images[s:e].to(device)
            rot  = TF.rotate(imgs, angle=float(angle_deg))
            rot_norm = (rot - CLASSIFIER_MEAN) / CLASSIFIER_STD
            probs, preds = F.softmax(classifier(rot_norm), dim=1).max(dim=1)
            mask = probs >= confidence_threshold
            if mask.sum() > 0:
                accepted = rot[mask].cpu()
                labels   = preds[mask].cpu()
                a_imgs.append(accepted)
                a_lbls.append(labels)
                a_base.append(torch.full((accepted.shape[0],), float(angle_deg)))
                for lbl in labels:
                    stats[angle_deg][lbl.item()] += 1

        if a_imgs:
            all_imgs.append(torch.cat(a_imgs))
            all_lbls.append(torch.cat(a_lbls))
            all_base.append(torch.cat(a_base))

    return (torch.cat(all_imgs), torch.cat(all_lbls),
            torch.cat(all_base), stats)

In [ ]:
@torch.no_grad()
def run_stage3(images, base_angles, batch_size=512):
    """Apply uniform random rotation θ~U(0,360). Final angle = (base + θ) mod 360."""
    N = images.shape[0]
    uniform_angles = torch.rand(N) * 360.0
    final_angles   = (base_angles + uniform_angles) % 360.0
    rotated = torch.empty_like(images)
    for s in tqdm(range(0, N, batch_size), desc='Stage 3: uniform rotation'):
        e = min(s + batch_size, N)
        for i, (img, ang) in enumerate(zip(images[s:e], uniform_angles[s:e])):
            rotated[s+i] = TF.rotate(img.unsqueeze(0), float(ang.item())).squeeze(0)
    return rotated, final_angles


def build_dataset(images, labels, base_angles, final_angles):
    """Package everything into a dict ready for saving / training."""
    imgs_norm  = images * 2.0 - 1.0                        # [0,1] → [-1,1]
    angles_rad = final_angles * (math.pi / 180.0)
    return {
        'images':      imgs_norm,
        'images_flat': imgs_norm.view(-1, 784),
        'angles_deg':  final_angles,
        'angle_cos':   torch.cos(angles_rad),
        'angle_sin':   torch.sin(angles_rad),
        'angle_vec':   torch.stack([torch.cos(angles_rad),
                                    torch.sin(angles_rad)], dim=1),
        'labels':      labels,
        'base_angles': base_angles,
    }

---
## 4. Build Thesis-Exact Dataset  (90° / 180° / 270°)

In [ ]:
THESIS_ANGLES = [90, 180, 270]
THESIS_PATH   = os.path.join(OUTPUT_DIR, 'mnist_rotation_thesis.pt')

if os.path.exists(THESIS_PATH):
    print(f'Loading existing thesis dataset from {THESIS_PATH}')
    thesis_data = torch.load(THESIS_PATH, map_location='cpu')
    # Rebuild stats from the saved data for analysis cells below
    thesis_stats = None   # will be reconstructed in the analysis section
else:
    aug_imgs_t, aug_lbls_t, aug_base_t, thesis_stats = run_stage2(
        raw_images, raw_labels, classifier,
        candidate_angles_deg=THESIS_ANGLES,
        confidence_threshold=CONFIDENCE_THRESHOLD,
    )
    final_imgs_t, final_angles_t = run_stage3(aug_imgs_t, aug_base_t)
    thesis_data = build_dataset(final_imgs_t, aug_lbls_t, aug_base_t, final_angles_t)
    torch.save(thesis_data, THESIS_PATH)
    print(f'Saved thesis dataset → {THESIS_PATH}')

N_thesis = thesis_data['images'].shape[0]
print(f'Total entries: {N_thesis:,}  (original 60,000 + {N_thesis-60000:,} augmented)')

---
## 5. Build Every-10-Degrees Dataset  (10°, 20°, ..., 350°)

In [ ]:
EVERY10_ANGLES = list(range(10, 360, 10))   # 35 candidate angles
EVERY10_PATH   = os.path.join(OUTPUT_DIR, 'mnist_rotation_every10.pt')

if os.path.exists(EVERY10_PATH):
    print(f'Loading existing every-10 dataset from {EVERY10_PATH}')
    every10_data  = torch.load(EVERY10_PATH, map_location='cpu')
    every10_stats = None
else:
    aug_imgs_e, aug_lbls_e, aug_base_e, every10_stats = run_stage2(
        raw_images, raw_labels, classifier,
        candidate_angles_deg=EVERY10_ANGLES,
        confidence_threshold=CONFIDENCE_THRESHOLD,
    )
    final_imgs_e, final_angles_e = run_stage3(aug_imgs_e, aug_base_e)
    every10_data = build_dataset(final_imgs_e, aug_lbls_e, aug_base_e, final_angles_e)
    torch.save(every10_data, EVERY10_PATH)
    print(f'Saved every-10 dataset → {EVERY10_PATH}')

N_every10 = every10_data['images'].shape[0]
print(f'Total entries: {N_every10:,}  (original 60,000 + {N_every10-60000:,} augmented)')

---
## 6. Analysis — Reconstruct Augmentation Counts from Saved Data

In [ ]:
def compute_stats_from_data(dataset, candidate_angles):
    """
    Reconstruct per-angle per-digit counts from a saved dataset.
    (base_angles == 0 are originals; the rest are augmented)
    """
    base = dataset['base_angles']   # [N]
    lbls = dataset['labels']        # [N]
    stats = {}
    for angle in candidate_angles:
        mask = (base == float(angle))
        counts = [(lbls[mask] == d).sum().item() for d in range(10)]
        stats[angle] = counts
    return stats

# If we loaded from disk rather than just computed, rebuild stats now
if thesis_stats is None:
    thesis_stats = compute_stats_from_data(thesis_data, THESIS_ANGLES)
if every10_stats is None:
    every10_stats = compute_stats_from_data(every10_data, EVERY10_ANGLES)

---
## 7. Table — Thesis-Exact Dataset (replicates thesis Table 4.5)

In [ ]:
def make_stats_df(stats, candidate_angles):
    """Turn stats dict into a styled DataFrame matching thesis Table 4.5."""
    rows = []
    for angle in candidate_angles:
        row = [f'{angle}°'] + stats[angle]
        row.append(sum(stats[angle]))
        rows.append(row)

    # Totals row
    totals = [sum(stats[a][d] for a in candidate_angles) for d in range(10)]
    rows.append(['Total'] + totals + [sum(totals)])

    cols = ['Angle'] + [str(d) for d in range(10)] + ['Total']
    df   = pd.DataFrame(rows, columns=cols).set_index('Angle')
    return df


thesis_df = make_stats_df(thesis_stats, THESIS_ANGLES)

def style_table(df, title):
    styled = (
        df.style
        .set_caption(title)
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '14px'), ('font-weight', 'bold'),
                       ('text-align', 'left'), ('padding-bottom', '8px')]},
            {'selector': 'th',
             'props': [('background-color', '#2c3e50'), ('color', 'white'),
                       ('text-align', 'center'), ('padding', '6px 10px')]},
            {'selector': 'td',
             'props': [('text-align', 'right'), ('padding', '4px 10px')]},
            {'selector': 'tr:last-child td',
             'props': [('font-weight', 'bold'), ('border-top', '2px solid #2c3e50')]},
            {'selector': 'tr:last-child th',
             'props': [('font-weight', 'bold'), ('border-top', '2px solid #2c3e50')]},
        ])
        .format(lambda x: f'{int(x):,}' if isinstance(x, (int, float)) else x)
        .highlight_max(axis=0, subset=df.columns[:-1],
                       props='background-color:#d5e8d4')
        .highlight_min(axis=0, subset=df.columns[:-1],
                       props='background-color:#f8cecc')
    )
    return styled

print('Thesis-Exact Dataset — Augmented Images per Digit per Rotation Angle')
print('(Replicates Table 4.5 in the thesis)')
print(f'Confidence threshold: >{CONFIDENCE_THRESHOLD}')
print()
style_table(thesis_df,
    f'Table 4.5 — Augmented images by digit and angle (confidence > {CONFIDENCE_THRESHOLD})')

In [ ]:
# Also print as plain text for easy copy-paste / LaTeX comparison
print(thesis_df.to_string())
print(f'\nOriginal images: 60,000')
print(f'Augmented entries: {int(thesis_df.loc["Total","Total"]):,}')
print(f'Total dataset size: {60000 + int(thesis_df.loc["Total","Total"]):,}')

---
## 8. Table — Every-10-Degrees Dataset

In [ ]:
every10_df = make_stats_df(every10_stats, EVERY10_ANGLES)

print('Every-10-Degrees Dataset — Augmented Images per Digit per Rotation Angle')
style_table(every10_df,
    f'Every-10° — Augmented images by digit and angle (confidence > {CONFIDENCE_THRESHOLD})')

In [ ]:
print(every10_df.to_string())
print(f'\nOriginal images: 60,000')
print(f'Augmented entries: {int(every10_df.loc["Total","Total"]):,}')
print(f'Total dataset size: {60000 + int(every10_df.loc["Total","Total"]):,}')

---
## 9. Visualisation — Augmented counts by angle (every-10 dataset)

In [ ]:
# ── Heatmap: digit × angle ─────────────────────────────────────
heat_data = np.array([every10_stats[a] for a in EVERY10_ANGLES])  # [35, 10]
heat_df   = pd.DataFrame(heat_data,
                          index=[f'{a}°' for a in EVERY10_ANGLES],
                          columns=[str(d) for d in range(10)])

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(heat_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Number of augmented images'})
ax.set_xlabel('Digit', fontsize=13)
ax.set_ylabel('Rotation angle', fontsize=13)
ax.set_title('Augmented images per digit per rotation angle\n'
             f'(confidence > {CONFIDENCE_THRESHOLD}, every-10° variant)',
             fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'heatmap_every10.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Bar chart: total augmented images per angle ────────────────
angle_totals_e10 = [sum(every10_stats[a]) for a in EVERY10_ANGLES]

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar([f'{a}°' for a in EVERY10_ANGLES], angle_totals_e10,
              color=sns.color_palette('Blues_d', len(EVERY10_ANGLES)))
ax.set_xlabel('Rotation angle', fontsize=12)
ax.set_ylabel('Number of augmented images', fontsize=12)
ax.set_title('Total augmented images per candidate rotation angle (every-10° variant)', fontsize=13)
ax.bar_label(bars, fmt='%d', fontsize=8, padding=2)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'bar_totals_every10.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'180° dominates (point symmetry): {every10_stats[180]} per digit')

In [ ]:
# ── Stacked bar: digit composition per angle ──────────────────
palette = sns.color_palette('tab10', 10)
fig, ax = plt.subplots(figsize=(14, 5))
bottom  = np.zeros(len(EVERY10_ANGLES))
x       = np.arange(len(EVERY10_ANGLES))

for d in range(10):
    vals = [every10_stats[a][d] for a in EVERY10_ANGLES]
    ax.bar(x, vals, bottom=bottom, color=palette[d], label=str(d), width=0.8)
    bottom += np.array(vals, dtype=float)

ax.set_xticks(x)
ax.set_xticklabels([f'{a}°' for a in EVERY10_ANGLES], rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Rotation angle', fontsize=12)
ax.set_ylabel('Augmented images', fontsize=12)
ax.set_title('Digit composition of augmented images per angle (every-10° variant)', fontsize=13)
ax.legend(title='Digit', bbox_to_anchor=(1.01, 1), loc='upper left', ncol=1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'stacked_bar_every10.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Augmented images per digit (totals, both datasets) ────────
thesis_digit_totals = [sum(thesis_stats[a][d] for a in THESIS_ANGLES) for d in range(10)]
every10_digit_totals= [sum(every10_stats[a][d] for a in EVERY10_ANGLES) for d in range(10)]

x = np.arange(10)
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, thesis_digit_totals, w, label='Thesis (90/180/270°)', color='#3498db')
b2 = ax.bar(x + w/2, every10_digit_totals, w, label='Every-10° (10–350°)', color='#e67e22')
ax.bar_label(b1, fmt='%d', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%d', fontsize=8, padding=2)
ax.set_xticks(x)
ax.set_xticklabels([str(d) for d in range(10)], fontsize=12)
ax.set_xlabel('Digit', fontsize=12)
ax.set_ylabel('Total augmented images', fontsize=12)
ax.set_title('Total augmented images per digit — thesis vs every-10° variant', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'digit_totals_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Visualisation — Sample Images from Both Datasets

In [ ]:
def show_samples(dataset, title, n_per_digit=5):
    """Show n_per_digit random samples for each digit, with their rotation angle."""
    imgs   = dataset['images']       # [N, 1, 28, 28]  in [-1,1]
    lbls   = dataset['labels']
    angles = dataset['angles_deg']
    base   = dataset['base_angles']

    fig, axes = plt.subplots(10, n_per_digit, figsize=(n_per_digit*1.5, 16))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    for digit in range(10):
        # prefer augmented entries (base > 0) to show interesting rotations
        mask_aug = (lbls == digit) & (base > 0)
        if mask_aug.sum() < n_per_digit:
            mask_aug = lbls == digit
        idxs = torch.where(mask_aug)[0]
        perm = torch.randperm(len(idxs))[:n_per_digit]
        chosen = idxs[perm]

        for j, idx in enumerate(chosen):
            ax = axes[digit][j]
            img = imgs[idx].squeeze().numpy()
            img = (img + 1) / 2   # [-1,1] → [0,1]
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
            ax.set_title(f'{angles[idx]:.0f}°', fontsize=7)
            ax.axis('off')
        axes[digit][0].set_ylabel(f'Digit {digit}', fontsize=9,
                                   rotation=0, labelpad=35, va='center')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'samples_{title.split("(")[0].strip().replace(" ","_").lower()}.png'),
                dpi=120, bbox_inches='tight')
    plt.show()

show_samples(thesis_data,   'Thesis dataset (90/180/270°)')
show_samples(every10_data,  'Every-10° dataset (10–350°)')

---
## 11. Angle Distribution Analysis

In [ ]:
# ── Polar histogram of final angles per dataset ────────────────
fig, axes = plt.subplots(1, 2, subplot_kw={'projection': 'polar'}, figsize=(12, 5))

for ax, data, name in zip(axes,
                           [thesis_data, every10_data],
                           ['Thesis (90/180/270°)', 'Every-10°']):
    angles_rad = (data['angles_deg'] * math.pi / 180).numpy()
    bins = np.linspace(0, 2*math.pi, 73)   # 5° bins
    counts, _ = np.histogram(angles_rad, bins=bins)
    theta = (bins[:-1] + bins[1:]) / 2
    ax.bar(theta, counts, width=2*math.pi/72, alpha=0.7, color='steelblue')
    ax.set_title(f'Final angle distribution\n{name}', pad=15, fontsize=11)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'polar_angle_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-digit angle distribution (every-10 dataset) ───────────
fig, axes = plt.subplots(2, 5, figsize=(16, 6), subplot_kw={'projection': 'polar'})
axes = axes.flatten()

for d in range(10):
    ax   = axes[d]
    mask = every10_data['labels'] == d
    ang  = (every10_data['angles_deg'][mask] * math.pi / 180).numpy()
    bins = np.linspace(0, 2*math.pi, 37)   # 10° bins
    counts, _ = np.histogram(ang, bins=bins)
    theta = (bins[:-1] + bins[1:]) / 2
    ax.bar(theta, counts, width=2*math.pi/36, alpha=0.75,
           color=sns.color_palette('tab10')[d])
    ax.set_title(f'Digit {d}  (n={mask.sum():,})', pad=10, fontsize=10)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.tick_params(labelsize=7)

fig.suptitle('Per-digit final angle distribution — every-10° dataset', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'polar_per_digit_every10.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Dataset Summary

In [ ]:
def print_summary(name, dataset, candidate_angles, stats):
    N     = dataset['images'].shape[0]
    n_aug = N - 60000
    aug_df = make_stats_df(stats, candidate_angles)

    print(f'{'='*60}')
    print(f' {name}')
    print(f'{'='*60}')
    print(f'  Original images : 60,000')
    print(f'  Augmented       : {n_aug:,}')
    print(f'  Total           : {N:,}')
    print(f'  Confidence thr  : > {CONFIDENCE_THRESHOLD}')
    print(f'  Candidate angles: {candidate_angles}')
    print()
    print(aug_df.to_string())
    print()

print_summary('THESIS-EXACT (90/180/270°)', thesis_data, THESIS_ANGLES, thesis_stats)
print_summary('EVERY-10-DEGREES (10–350°)', every10_data, EVERY10_ANGLES, every10_stats)

In [ ]:
# ── How to use these datasets in the training scripts ──────────
print("""
=== USAGE IN TRAINING SCRIPTS ===

from torch.utils.data import DataLoader, TensorDataset

# Load the saved dataset
data = torch.load('data/mnist_rotation/mnist_rotation_thesis.pt')  # or every10

# For conditional model  (yields image_flat [B,784], angle_vec [B,2])
ds_cond   = TensorDataset(data['images_flat'], data['angle_vec'])
loader_cond = DataLoader(ds_cond, batch_size=256, shuffle=True)

# For unconditional UNet (yields image [B,1,28,28])
ds_uncond   = TensorDataset(data['images'])
loader_uncond = DataLoader(ds_uncond, batch_size=256, shuffle=True)
""")